# FCFF教学案例（完成数据获取与财务预测两阶段工作）


可以把整份代码理解成三步：

1. 从 RiceQuant 读取历史财务数据和市场数据。
2. 根据用户输入的假设，预测未来的利润、资产负债表和现金流。


## 一、准备工作

这一部分导入依赖库，并定义脚本会反复用到的常量。

- `STATEMENT_UNIT` 表示把财务报表统一换算成“百万元”
- `SHARE_UNIT` 表示把总股本统一换算成“百万股”
- `INCOME_FIELDS` 和 `BALANCE_FIELDS` 是从 RiceQuant 抓取的历史字段列表

In [2]:
import copy
import pandas as pd
from pkufinlab import rq_init
rq=rq_init()

STATEMENT_UNIT = 1_000_000.0
SHARE_UNIT = 1_000_000.0
INCOME_FIELDS = ["revenue", "profit_before_tax", "minority_profit", "net_profit", "net_profit_parent_company", "ebit"]
BALANCE_FIELDS = ["cash_equivalent", "financial_asset_held_for_trading", "current_assets", "bill_receivable", "net_accts_receivable", "other_accts_receivable", "inventory", "deferred_expense", "other_current_assets", "long_term_equity_investment", "net_long_term_equity_investment", "real_estate_investment", "net_fixed_assets", "construction_in_progress", "fixed_asset_to_be_disposed", "intangible_assets", "long_term_deferred_expenses", "deferred_income_tax_assets", "other_non_current_assets", "short_term_loans", "notes_payable", "accts_payable", "long_term_liabilities_due_one_year", "non_current_liability_due_one_year", "other_current_liabilities", "current_liabilities", "long_term_loans", "bond_payable", "long_term_payable", "deferred_income_tax_liabilities", "deferred_revenue", "other_non_current_liabilities", "total_assets", "minority_interest", "equity_parent_company", "total_equity"]


## 二、基础工具函数

这部分函数本身不直接做估值，而是负责一些通用工作：

- 打印日志，方便看清代码运行到哪一步
- 处理日期和股票代码格式
- 生成历史回看区间
- 从历史表中读取最后一期数值
- 用备用字段补齐缺失数据

In [3]:
def log(msg="", verbose=True):
    """打印普通日志。

    参数：
    - `msg`：日志文本。
    - `verbose`：是否输出日志。
    """
    if verbose:
        print(msg)


def title(msg, verbose=True):
    """打印阶段标题。

    参数：
    - `msg`：标题文本。
    - `verbose`：是否输出日志。
    """
    if verbose:
        line = "=" * max(len(msg), 24)
        print("\n" + line)
        print(msg)
        print(line)


def subtitle(msg, verbose=True):
    """打印阶段内小标题。

    参数：
    - `msg`：小标题文本。
    - `verbose`：是否输出日志。
    """
    if verbose:
        print(f"\n[{msg}]")


def init_rq():
    """初始化 RiceQuant 连接。

    参数：
    - 无。函数内部使用文件顶部写死的账号信息。
    """
    import rqdatac as rq
    if not rq.initialized():
        from pkufinlab import rq_init
        rq=rq_init()


def ts(x):
    """把日期输入统一转成 `pandas.Timestamp`。

    参数：
    - `x`：字符串或可被 pandas 识别的日期对象。
    """
    return pd.Timestamp(x).normalize()


def rq_code(code):
    """把证券代码转成 RiceQuant 格式。

    参数：
    - `code`：如 `000012.SZ`、`600000.SH`。
    """
    code = code.strip().upper()
    if code.endswith(".XSHE") or code.endswith(".XSHG"):
        return code
    if code.endswith(".SZ"):
        return code.replace(".SZ", ".XSHE")
    if code.endswith(".SH"):
        return code.replace(".SH", ".XSHG")
    raise ValueError("股票代码格式不支持")


def latest_hist_year(report_date, annual_report_released):
    """计算估值观察日下可见的最近完整历史年份。

    参数：
    - `report_date`：估值观察日。
    - `annual_report_released`：最近一年年报是否已披露。
    """
    return ts(report_date).year - 1 if annual_report_released else ts(report_date).year - 2


def hist_years(report_date, annual_report_released, lookback_years):
    """生成历史财务回看年份列表。

    参数：
    - `report_date`：估值观察日。
    - `annual_report_released`：最近一年年报是否已披露。
    - `lookback_years`：向前回看几年。
    """
    y = latest_hist_year(report_date, annual_report_released)
    return list(range(y - lookback_years + 1, y + 1))


def fetch_annual_pit(order_book_id, report_date, annual_report_released, fields, lookback_years, market="cn"):
    """按 PIT 口径抓取历史年报数据并统一到百万元。

    参数：
    - `order_book_id`：RiceQuant 证券代码。
    - `report_date`：估值观察日。
    - `annual_report_released`：最近一年年报是否已披露。
    - `fields`：要抓取的字段列表。
    - `lookback_years`：历史回看年数。
    - `market`：市场代码。
    """
    years = hist_years(report_date, annual_report_released, lookback_years)
    raw = rq.get_pit_financials_ex(
        [order_book_id],
        fields,
        start_quarter=f"{years[0]}q4",
        end_quarter=f"{years[-1]}q4",
        date=ts(report_date).strftime("%Y-%m-%d"),
        statements="latest",
        market=market,
    )
    if raw is None or raw.empty:
        raise ValueError("RiceQuant 未返回 PIT 财务数据")
    df = raw.reset_index()
    df = df[df["quarter"].isin([f"{y}q4" for y in years])].copy()
    df["year"] = df["quarter"].str[:4].astype(int)
    df = df.sort_values(["year", "info_date", "rice_create_tm"]).drop_duplicates("year", keep="last")
    df = df.set_index("year")[fields].reindex(years).astype(float) / STATEMENT_UNIT
    df.index.name = "year"
    return df


def fetch_close(order_book_id, report_date, market="cn"):
    """抓取估值观察日的收盘价。

    参数：
    - `order_book_id`：RiceQuant 证券代码。
    - `report_date`：估值观察日。
    - `market`：市场代码。
    """
    p = rq.get_price(order_book_id, start_date=ts(report_date).strftime("%Y-%m-%d"), end_date=ts(report_date).strftime("%Y-%m-%d"), fields=["close"], adjust_type="none", market=market)
    return float(p.squeeze())


def fetch_shares_mn(order_book_id, report_date, market="cn"):
    """抓取估值观察日总股本并换算成百万股。

    参数：
    - `order_book_id`：RiceQuant 证券代码。
    - `report_date`：估值观察日。
    - `market`：市场代码。
    """
    s = rq.get_shares(order_book_id, start_date=ts(report_date).strftime("%Y-%m-%d"), end_date=ts(report_date).strftime("%Y-%m-%d"), fields=["total"], market=market)
    return float(s.squeeze()) / SHARE_UNIT


def fetch_name(order_book_id, market="cn"):
    """抓取证券简称。

    参数：
    - `order_book_id`：RiceQuant 证券代码。
    - `market`：市场代码。
    """
    ins = rq.instruments(order_book_id, market=market)
    if isinstance(ins, list):
        ins = ins[0]
    return getattr(ins, "symbol", order_book_id)


def lastv(df, col, default=0.0):
    """读取某列最后一期的值，不存在时返回默认值。

    参数：
    - `df`：数据表。
    - `col`：字段名。
    - `default`：默认值。
    """
    if col not in df.columns:
        return float(default)
    v = df[col].iloc[-1]
    return 0.0 if pd.isna(v) else float(v)


def coalesce(primary, fallback):
    """优先使用主序列，缺失时用备用序列补齐。

    参数：
    - `primary`：主序列。
    - `fallback`：备用序列。
    """
    if primary is None:
        return None if fallback is None else fallback.astype(float)
    if fallback is None:
        return primary.astype(float)
    s = primary.astype(float).copy()
    f = fallback.astype(float)
    mask = s.isna() | ((s.abs() < 1e-12) & f.notna() & (f.abs() > 1e-12))
    s.loc[mask] = f.loc[mask]
    return s


## 三、第一阶段：数据获取

这一部分对应 FCFF 估值的起点。

- 根据估值日确定“当前能看到的历史年份”
- 从 RiceQuant 读取历史利润表和资产负债表
- 读取估值日的收盘价、总股本和证券简称

最后输出一个 `source` 字典，作为后续预测和估值的基础输入。

In [4]:
def get_source(cfg, verbose=True):
    """完成第一阶段数据获取。

    参数：
    - `cfg`：`USER_INPUTS["data_import"]` 对应的字典。
    - `verbose`：是否打印日志。
    """
    title("Stage 1/3: Data Acquisition", verbose)
    subtitle("1.1 Input Context", verbose)
    log(f"stock_code={cfg['stock_code']}, report_date={cfg['report_date']}, annual_report_released={cfg['annual_report_released']}", verbose)
    if cfg.get("statement_scope", "合并报表") != "合并报表":
        raise ValueError("当前脚本只支持合并报表口径")
    init_rq()
    order_book_id = rq_code(cfg["stock_code"])
    years = hist_years(cfg["report_date"], cfg["annual_report_released"], cfg.get("lookback_years", 5))
    income = fetch_annual_pit(order_book_id, cfg["report_date"], cfg["annual_report_released"], INCOME_FIELDS, cfg.get("lookback_years", 5), cfg.get("market", "cn"))
    balance = fetch_annual_pit(order_book_id, cfg["report_date"], cfg["annual_report_released"], BALANCE_FIELDS, cfg.get("lookback_years", 5), cfg.get("market", "cn"))
    if "long_term_liabilities_due_one_year" in balance.columns or "non_current_liability_due_one_year" in balance.columns:
        balance["long_term_liabilities_due_one_year"] = coalesce(balance["long_term_liabilities_due_one_year"] if "long_term_liabilities_due_one_year" in balance.columns else None, balance["non_current_liability_due_one_year"] if "non_current_liability_due_one_year" in balance.columns else None)
    if "long_term_equity_investment" in balance.columns or "net_long_term_equity_investment" in balance.columns:
        balance["long_term_equity_investment"] = coalesce(balance["long_term_equity_investment"] if "long_term_equity_investment" in balance.columns else None, balance["net_long_term_equity_investment"] if "net_long_term_equity_investment" in balance.columns else None)
    market = pd.Series({"order_book_id": order_book_id, "security_name": fetch_name(order_book_id, cfg.get("market", "cn")), "report_date": ts(cfg["report_date"]), "close_price": fetch_close(order_book_id, cfg["report_date"], cfg.get("market", "cn")), "total_shares_mn": fetch_shares_mn(order_book_id, cfg["report_date"], cfg.get("market", "cn"))})
    subtitle("1.2 Historical Window", verbose)
    log(f"historical_years={years}", verbose)
    subtitle("1.3 Market Snapshot", verbose)
    log(market[["security_name", "close_price", "total_shares_mn"]].to_string(), verbose)
    log("[Stage 1/3 Completed] Data acquisition finished", verbose)
    return {"historical_years": years, "income": income, "balance": balance, "market": market}

## 四、第二阶段开始前：整理用户输入

FCFF 模型里有很多“人为设定的假设”，例如：

- 未来收入增速
- 成本率和费用率
- 存货和应收应付周转天数
- 资本开支和借款安排

这部分代码的作用，就是把用户输入的标量或列表，统一整理成按年份排列的 `Series` 或 `DataFrame`，让后面的预测函数更容易计算。

In [5]:
def series(val, years, name):
    """把标量或列表整理成按年份索引的 Series。

    参数：
    - `val`：标量、列表或 `Series`。
    - `years`：预测年份列表。
    - `name`：变量名，用于报错提示。
    """
    if isinstance(val, pd.Series):
        s = val.astype(float).copy()
        s.index = s.index.astype(int)
        s = s.reindex(years)
    elif isinstance(val, (list, tuple)):
        if len(val) != len(years):
            raise ValueError(f"{name} 长度不匹配")
        s = pd.Series(list(val), index=years, dtype=float)
    else:
        s = pd.Series(float(val), index=years, dtype=float)
    if s.isna().any():
        raise ValueError(f"{name} 存在缺失值")
    return s.astype(float)


def resolve_inputs(payload, years):
    """把用户输入拆分成各模块的预测参数表。

    参数：
    - `payload`：完整输入字典。
    - `years`：预测年份列表。
    """
    fa = payload["forecast_assumptions"]
    op, wc, cp, db = fa["operating"], fa["working_capital"], fa["capex"], fa["debt"]
    eq, ap = fa.get("equity", {}), fa.get("asset_policy", {})
    op_df = pd.DataFrame({k: series(v, years, k) for k, v in op.items()})
    wc_df = pd.DataFrame({k: series(v, years, k) for k, v in wc.items()})
    cp_df = pd.DataFrame({"fixed_asset_new_project_investment": series(cp["fixed_asset_new_project_investment"], years, "fixed_asset_new_project_investment"), "intangible_capex": series(cp["intangible_capex"], years, "intangible_capex"), "fixed_asset_disposal": series(cp.get("fixed_asset_disposal", 0.0), years, "fixed_asset_disposal"), "cip_loan_coverage_ratio": series(cp.get("cip_loan_coverage_ratio", 0.0), years, "cip_loan_coverage_ratio"), "cip_completion_ratio": series(cp.get("cip_completion_ratio", 0.7), years, "cip_completion_ratio"), "renovation_investment_ratio": series(cp.get("renovation_investment_ratio", 0.05), years, "renovation_investment_ratio"), "impairment_ratio": series(cp.get("impairment_ratio", 0.0), years, "impairment_ratio"), "renovation_transfer_ratio": series(cp.get("renovation_transfer_ratio", 1.0), years, "renovation_transfer_ratio")})
    db_df = pd.DataFrame({"min_cash_balance": series(db["min_cash_balance"], years, "min_cash_balance"), "cash_interest_rate": series(db["cash_interest_rate"], years, "cash_interest_rate"), "min_revolver_balance": series(db["min_revolver_balance"], years, "min_revolver_balance"), "revolver_interest_rate": series(db["revolver_interest_rate"], years, "revolver_interest_rate"), "bond_increment": series(db["bond_increment"], years, "bond_increment"), "bond_interest_rate": series(db["bond_interest_rate"], years, "bond_interest_rate"), "long_term_loan_increment": series(db["long_term_loan_increment"], years, "long_term_loan_increment"), "long_term_loan_interest_rate": series(db["long_term_loan_interest_rate"], years, "long_term_loan_interest_rate"), "capitalized_interest_rate": series(db.get("capitalized_interest_rate", db["bond_interest_rate"]), years, "capitalized_interest_rate")})
    eq_df = pd.DataFrame({"share_split_multiplier": series(eq.get("share_split_multiplier", 1.0), years, "share_split_multiplier"), "equity_issue_price": series(eq.get("equity_issue_price", 0.0), years, "equity_issue_price"), "equity_issue_shares": series(eq.get("equity_issue_shares", 0.0), years, "equity_issue_shares"), "convertible_price": series(eq.get("convertible_price", 0.0), years, "convertible_price"), "convertible_shares": series(eq.get("convertible_shares", 0.0), years, "convertible_shares")})
    eq_df["equity_issue_amount"] = eq_df["equity_issue_price"] * eq_df["equity_issue_shares"]
    eq_df["debt_to_equity_reduction"] = eq_df["convertible_price"] * eq_df["convertible_shares"]
    ap_df = pd.DataFrame({"short_term_investment_balance": series(ap.get("short_term_investment_balance", 0.0), years, "short_term_investment_balance")})
    scalars = {"existing_fixed_asset_remaining_life": float(cp.get("existing_fixed_asset_remaining_life", 10.0)), "new_fixed_asset_life": float(cp.get("new_fixed_asset_life", 15.0)), "intangible_amortization_life": float(cp.get("intangible_amortization_life", 25.0)), "opening_cip_interest_balance": float(cp.get("opening_cip_interest_balance", 0.0))}
    return {"operating": op_df, "working_capital": wc_df, "capex": cp_df, "debt": db_df, "equity": eq_df, "asset_policy": ap_df, "scalars": scalars}

## 五、第二阶段：财务预测

这部分是整个脚本里最核心的部分。可以把它理解成“把一套未来假设，逐年翻译成财务报表”。

这里分成几个模块：

- `share_schedule`：预测总股本
- `capex_schedule`：预测资本开支、折旧、摊销、在建工程
- `income_schedule`：预测收入、EBIT、净利润、分红
- `wc_schedule`：预测营运资本占用

这些函数合在一起，构成未来经营结果的主体。

In [6]:
def share_schedule(src, eq, years):
    """生成股本预测表。

    参数：
    - `src`：第一阶段源数据。
    - `eq`：股本假设表。
    - `years`：预测年份列表。
    """
    prev = float(src["market"]["total_shares_mn"])
    rows = []
    for i, y in enumerate(years):
        r = eq.loc[y].to_dict()
        total = prev if i == 0 else prev * r["share_split_multiplier"] + r["equity_issue_shares"] + r["convertible_shares"]
        r.update({"year": y, "total_shares_mn": float(total)})
        rows.append(r)
        prev = float(total)
    return pd.DataFrame(rows).set_index("year")


def capex_schedule(src, rs, years, capitalized_interest=None):
    """生成资本开支、转固、折旧和摊销预测表。

    参数：
    - `src`：第一阶段源数据。
    - `rs`：整理后的预测假设。
    - `years`：预测年份列表。
    - `capitalized_interest`：资本化利息序列。
    """
    b, op, cp, sc = src["balance"], rs["operating"], rs["capex"], rs["scalars"]
    if capitalized_interest is None:
        capitalized_interest = pd.Series(0.0, index=years, dtype=float)
    life0, life1 = sc["existing_fixed_asset_remaining_life"], sc["new_fixed_asset_life"]
    open_fix_net = lastv(b, "net_fixed_assets")
    impair0 = float(cp["impairment_ratio"].iloc[0])
    open_fix_book = open_fix_net if abs(1 - impair0) < 1e-12 else open_fix_net / (1 - impair0)
    open_cip = lastv(b, "construction_in_progress") + lastv(b, "engineer_material")
    open_int = lastv(b, "intangible_assets")
    rows, cohorts = [], []
    cur_cip, cur_fix, dep_old = open_cip, open_fix_book, open_fix_book / life0
    for i, y in enumerate(years):
        new_proj, int_capex, disposal, cap_i = float(cp.loc[y, "fixed_asset_new_project_investment"]), float(cp.loc[y, "intangible_capex"]), float(cp.loc[y, "fixed_asset_disposal"]), float(capitalized_interest.loc[y])
        ren_ratio, ren_transfer_ratio = float(cp.loc[y, "renovation_investment_ratio"]), float(cp.loc[y, "renovation_transfer_ratio"])
        ren, cons = cur_fix * ren_ratio, new_proj - cur_fix * ren_ratio
        ren_transfer = ren * ren_transfer_ratio
        cip_transfer = (cur_cip + cons) * float(cp.loc[y, "cip_completion_ratio"])
        interest_transfer = (sc["opening_cip_interest_balance"] + cap_i) * float(cp.loc[y, "cip_completion_ratio"]) if i == 0 else 0.0
        fix_inc = ren_transfer + cip_transfer + interest_transfer
        cohorts.append((y, fix_inc))
        dep = dep_old if i < int(life0) else 0.0
        for cy, amt in cohorts:
            age = y - cy
            if age < int(life1):
                dep += (amt / life1) / 2 if age == 0 else amt / life1
        end_cip = cur_cip + ren + cons + cap_i - fix_inc
        fix_book = cur_fix + fix_inc - dep + float(op.loc[y, "impairment_and_fair_value"])
        rows.append({"year": y, "opening_cip": cur_cip, "renovation_investment": ren, "construction_investment": cons, "capitalized_interest": cap_i, "fixed_asset_increase": fix_inc, "depreciation": dep, "ending_cip": end_cip, "fixed_asset_book_value": fix_book, "fixed_asset_disposal": disposal, "intangible_capex": int_capex})
        cur_cip, cur_fix = end_cip, fix_book
    s = pd.DataFrame(rows).set_index("year")
    s["impairment_reserve"] = s["fixed_asset_book_value"] * cp["impairment_ratio"]
    s["fixed_asset_net_value"] = s["fixed_asset_book_value"] - s["impairment_reserve"]
    s["amortization"] = pd.Series(open_int / sc["intangible_amortization_life"], index=years, dtype=float)
    cur, end_int = open_int, []
    for y in years:
        cur = cur - float(s.loc[y, "amortization"]) + float(s.loc[y, "intangible_capex"])
        end_int.append(cur)
    s["ending_intangible"] = pd.Series(end_int, index=years, dtype=float)
    s["cash_capex"] = s["renovation_investment"] + s["construction_investment"] + s["capitalized_interest"] + s["intangible_capex"]
    s["capital_investment_cf"] = -s["cash_capex"]
    return s


def income_schedule(src, rs, capex, shares, years, interest_expense=None, interest_income=None):
    """生成损益表预测。

    参数：
    - `src`：第一阶段源数据。
    - `rs`：整理后的预测假设。
    - `capex`：资本开支预测表。
    - `shares`：股本预测表。
    - `years`：预测年份列表。
    - `interest_expense`：利息支出序列。
    - `interest_income`：利息收入序列。
    """
    hist, op = src["income"], rs["operating"]
    interest_expense = pd.Series(0.0, index=years, dtype=float) if interest_expense is None else series(interest_expense, years, "interest_expense")
    interest_income = pd.Series(0.0, index=years, dtype=float) if interest_income is None else series(interest_income, years, "interest_income")
    prev_rev, prev_pt, prev_min = lastv(hist, "revenue"), lastv(hist, "profit_before_tax"), lastv(hist, "minority_profit")
    rows = []
    for y in years:
        rev = prev_rev * (1 + float(op.loc[y, "revenue_growth"]))
        cost, sales_tax = rev * float(op.loc[y, "operating_cost_ratio"]), rev * float(op.loc[y, "sales_tax_ratio"])
        sell, ga, rnd = rev * float(op.loc[y, "selling_ratio"]), rev * float(op.loc[y, "ga_ratio"]), rev * float(op.loc[y, "rnd_ratio"])
        dep, amo = float(capex.loc[y, "depreciation"]), float(capex.loc[y, "amortization"])
        ebitda = (rev - cost) - sales_tax - sell - ga - rnd + dep
        ebit = ebitda - dep - amo
        pretax = ebit + float(op.loc[y, "fx_gain_loss"]) - float(interest_expense.loc[y]) + float(interest_income.loc[y]) + float(op.loc[y, "other_business_profit"]) + float(op.loc[y, "impairment_and_fair_value"]) + float(op.loc[y, "investment_income"]) + float(op.loc[y, "non_operating_pnl"])
        tax = pretax * float(op.loc[y, "income_tax_rate"])
        minority = 0.0 if abs(prev_pt) < 1e-12 else prev_min * pretax / prev_pt
        net = pretax - tax - minority
        div = max(net * float(op.loc[y, "dividend_payout_ratio"]), 0.0)
        rows.append({"year": y, "revenue": rev, "operating_cost": cost, "selling_expense": sell, "ga_expense_ex_amort": ga, "ebit": ebit, "pretax_profit": pretax, "minority_profit": minority, "net_profit": net, "cash_dividend": div})
        prev_rev, prev_pt, prev_min = rev, pretax, minority
    return pd.DataFrame(rows).set_index("year")


def wc_schedule(src, rs, inc, capex, years):
    """生成营运资本及其现金流调整项预测。

    参数：
    - `src`：第一阶段源数据。
    - `rs`：整理后的预测假设。
    - `inc`：损益表预测。
    - `capex`：资本开支预测表。
    - `years`：预测年份列表。
    """
    b, wc, op = src["balance"], rs["working_capital"], rs["operating"]
    prev_rec = lastv(b, "bill_receivable") + lastv(b, "net_accts_receivable") + lastv(b, "other_accts_receivable")
    prev_inv, prev_pay, prev_acc = lastv(b, "inventory"), lastv(b, "notes_payable") + lastv(b, "accts_payable"), 0.0
    opening_revolver = lastv(b, "short_term_loans") + lastv(b, "long_term_liabilities_due_one_year")
    hist_ca = lastv(b, "current_assets", lastv(b, "cash_equivalent") + lastv(b, "financial_asset_held_for_trading") + prev_rec + prev_inv + lastv(b, "deferred_expense") + lastv(b, "other_current_assets"))
    prev_oca = hist_ca - lastv(b, "cash_equivalent") - lastv(b, "financial_asset_held_for_trading") - prev_rec - prev_inv
    hist_cl = lastv(b, "current_liabilities", opening_revolver + prev_pay + prev_acc + lastv(b, "other_current_liabilities"))
    prev_ocl = hist_cl - opening_revolver - prev_pay - prev_acc
    prev_ltei = lastv(b, "long_term_equity_investment")
    prev_oltl = lastv(b, "long_term_payable") + lastv(b, "deferred_income_tax_liabilities") + lastv(b, "deferred_revenue") + lastv(b, "other_non_current_liabilities")
    impair0 = float(rs["capex"]["impairment_ratio"].iloc[0])
    prev_imp = 0.0 if abs(1 - impair0) < 1e-12 else lastv(b, "net_fixed_assets") * impair0 / (1 - impair0)
    prev_disposal = lastv(b, "fixed_asset_to_be_disposed")
    rows = []
    for y in years:
        rev = float(inc.loc[y, "revenue"])
        cost_sales = float(inc.loc[y, "operating_cost"]) + rev * float(op.loc[y, "sales_tax_ratio"]) - float(capex.loc[y, "depreciation"])
        op_exp = rev * float(op.loc[y, "selling_ratio"]) + rev * float(op.loc[y, "ga_ratio"])
        rec, inv = rev * float(wc.loc[y, "ar_days"]) / 365.0, cost_sales * float(wc.loc[y, "inventory_days"]) / 365.0
        oca, pay = rev * float(wc.loc[y, "other_current_asset_ratio"]), cost_sales * float(wc.loc[y, "ap_days"]) / 365.0
        acc, ocl = op_exp * float(wc.loc[y, "accrued_expense_ratio"]), (cost_sales + op_exp) * float(wc.loc[y, "other_current_liability_ratio"])
        ltei = prev_ltei + float(wc.loc[y, "long_term_equity_investment_increase"])
        oltl = prev_oltl + float(wc.loc[y, "other_long_term_liability_increase"])
        imp, disposal = float(capex.loc[y, "impairment_reserve"]), float(capex.loc[y, "fixed_asset_disposal"])
        cf = (prev_rec - rec) + (prev_inv - inv) + (prev_oca - oca) + (imp - prev_imp) + (pay - prev_pay) + (acc - prev_acc) + (ocl - prev_ocl) + (oltl - prev_oltl) + (prev_disposal - disposal)
        rows.append({"year": y, "receivables": rec, "inventory": inv, "other_current_assets": oca, "payables": pay, "accrued_expense": acc, "other_current_liabilities": ocl, "long_term_equity_investment": ltei, "other_long_term_liabilities": oltl, "working_capital_balance": rec + inv + oca - pay - acc - ocl + ltei - oltl, "net_working_capital_change_cf": cf})
        prev_rec, prev_inv, prev_oca, prev_pay, prev_acc, prev_ocl, prev_ltei, prev_oltl, prev_imp, prev_disposal = rec, inv, oca, pay, acc, ocl, ltei, oltl, imp, disposal
    return pd.DataFrame(rows).set_index("year")

## 六、第二阶段继续：现金流、债务与资产负债表

前面已经预测了经营结果，这里继续把它补全为完整的估值输入：

- `cashflow_schedule`：把利润表和营运资本变化转成现金流
- `debt_schedule`：根据最低现金保有量等约束，反推出借款和利息
- `bs_schedule`：生成估值需要的简化资产负债表
- `forecast`：把第二阶段全部串起来，并做联立求解

其中债务、现金和利息之间是互相影响的，所以脚本会迭代求解，直到结果收敛。

In [7]:
def cashflow_schedule(rs, shares, inc, wc, capex, years, revolver_change=None):
    """生成经营、投资、融资现金流预测。

    参数：
    - `rs`：整理后的预测假设。
    - `shares`：股本预测表。
    - `inc`：损益表预测。
    - `wc`：营运资本预测表。
    - `capex`：资本开支预测表。
    - `years`：预测年份列表。
    - `revolver_change`：循环贷款变动序列。
    """
    ap, debt, wci, op = rs["asset_policy"], rs["debt"], rs["working_capital"], rs["operating"]
    prev_sti = 0.0
    revolver_change = pd.Series(0.0, index=years, dtype=float) if revolver_change is None else series(revolver_change, years, "revolver_change")
    rows = []
    for y in years:
        sti = float(ap.loc[y, "short_term_investment_balance"])
        minority_retained = float(inc.loc[y, "minority_profit"]) * (1 - float(wci.loc[y, "minority_dividend_ratio"]))
        da = float(capex.loc[y, "depreciation"]) + float(capex.loc[y, "amortization"])
        ocf = float(inc.loc[y, "net_profit"]) + minority_retained - float(op.loc[y, "impairment_and_fair_value"]) + da + float(wc.loc[y, "net_working_capital_change_cf"])
        icf = (prev_sti - sti) - float(wci.loc[y, "long_term_equity_investment_increase"]) + float(capex.loc[y, "capital_investment_cf"])
        fcf_pre = float(shares.loc[y, "equity_issue_amount"]) + float(debt.loc[y, "long_term_loan_increment"]) + float(debt.loc[y, "bond_increment"]) - float(inc.loc[y, "cash_dividend"])
        rows.append({"year": y, "operating_cf": ocf, "investing_cf": icf, "financing_cf_pre_revolver": fcf_pre, "revolver_change": float(revolver_change.loc[y]), "financing_cf": fcf_pre + float(revolver_change.loc[y]), "cash_net_change": ocf + icf + fcf_pre + float(revolver_change.loc[y])})
        prev_sti = sti
    return pd.DataFrame(rows).set_index("year")


def debt_schedule(src, rs, shares, cfs, capex, years):
    """生成现金、循环贷款、长期借款和债券预测。

    参数：
    - `src`：第一阶段源数据。
    - `rs`：整理后的预测假设。
    - `shares`：股本预测表。
    - `cfs`：现金流预测表。
    - `capex`：资本开支预测表。
    - `years`：预测年份列表。
    """
    b, debt, cp = src["balance"], rs["debt"], rs["capex"]
    cur_cash = lastv(b, "cash_equivalent")
    cur_rev = lastv(b, "short_term_loans") + lastv(b, "long_term_liabilities_due_one_year")
    cur_ltl, cur_bond = lastv(b, "long_term_loans"), lastv(b, "bond_payable")
    rows = []
    for y in years:
        min_cash, min_rev = float(debt.loc[y, "min_cash_balance"]), float(debt.loc[y, "min_revolver_balance"])
        cash_gap = cur_cash + float(cfs.loc[y, "operating_cf"]) + float(cfs.loc[y, "investing_cf"]) + float(cfs.loc[y, "financing_cf_pre_revolver"]) - min_cash
        repay = cur_rev - min_rev
        rev_change = -min(cash_gap, repay)
        end_rev = cur_rev + rev_change
        rev_int = (cur_rev + end_rev) / 2 * float(debt.loc[y, "revolver_interest_rate"])
        ltl_change = float(debt.loc[y, "long_term_loan_increment"])
        end_ltl = cur_ltl + ltl_change
        ltl_int = (cur_ltl + end_ltl) / 2 * float(debt.loc[y, "long_term_loan_interest_rate"])
        bond_change = float(debt.loc[y, "bond_increment"]) - float(shares.loc[y, "debt_to_equity_reduction"])
        end_bond = cur_bond + bond_change
        bond_int = (cur_bond + end_bond) / 2 * float(debt.loc[y, "bond_interest_rate"])
        cap_i = ((float(capex.loc[y, "opening_cip"]) + float(capex.loc[y, "ending_cip"])) / 2) * float(cp.loc[y, "cip_loan_coverage_ratio"]) * float(debt.loc[y, "capitalized_interest_rate"])
        end_cash = min_cash + max(cash_gap - repay, 0.0)
        cash_int = (cur_cash + end_cash) / 2 * float(debt.loc[y, "cash_interest_rate"])
        rows.append({"year": y, "cash_gap_after_min_cash": cash_gap, "revolver_change": rev_change, "ending_revolver": end_rev, "ending_long_term_loans": end_ltl, "ending_bonds": end_bond, "capitalized_interest": cap_i, "interest_expense": rev_int + ltl_int - cap_i + bond_int, "ending_cash": end_cash, "cash_interest_income": cash_int})
        cur_cash, cur_rev, cur_ltl, cur_bond = end_cash, end_rev, end_ltl, end_bond
    return pd.DataFrame(rows).set_index("year")


def bs_schedule(src, rs, shares, inc, wc, capex, debt, years):
    """生成估值所需的简化资产负债表预测。

    参数：
    - `src`：第一阶段源数据。
    - `rs`：整理后的预测假设。
    - `shares`：股本预测表。
    - `inc`：损益表预测。
    - `wc`：营运资本预测表。
    - `capex`：资本开支预测表。
    - `debt`：债务预测表。
    - `years`：预测年份列表。
    """
    b, ap, wci = src["balance"], rs["asset_policy"], rs["working_capital"]
    hist_rec = lastv(b, "bill_receivable") + lastv(b, "net_accts_receivable") + lastv(b, "other_accts_receivable")
    hist_ca = lastv(b, "current_assets", lastv(b, "cash_equivalent") + lastv(b, "financial_asset_held_for_trading") + hist_rec + lastv(b, "inventory") + lastv(b, "deferred_expense") + lastv(b, "other_current_assets"))
    hist_fix = lastv(b, "net_fixed_assets") + lastv(b, "construction_in_progress") + lastv(b, "fixed_asset_to_be_disposed")
    hist_total = lastv(b, "total_assets")
    const_olta = hist_total - hist_ca - hist_fix - lastv(b, "intangible_assets") - lastv(b, "long_term_equity_investment") if abs(hist_total) > 1e-12 else lastv(b, "real_estate_investment") + lastv(b, "deferred_income_tax_assets") + lastv(b, "other_non_current_assets") + lastv(b, "long_term_deferred_expenses")
    cur_eq, cur_mi = lastv(b, "equity_parent_company"), lastv(b, "minority_interest")
    rows = []
    for y in years:
        cash, sti = float(debt.loc[y, "ending_cash"]), float(ap.loc[y, "short_term_investment_balance"])
        rec, inv, oca = float(wc.loc[y, "receivables"]), float(wc.loc[y, "inventory"]), float(wc.loc[y, "other_current_assets"])
        fix_net, cip, disposal = float(capex.loc[y, "fixed_asset_net_value"]), float(capex.loc[y, "ending_cip"]), float(capex.loc[y, "fixed_asset_disposal"])
        intang, ltei = float(capex.loc[y, "ending_intangible"]), float(wc.loc[y, "long_term_equity_investment"])
        total_assets = cash + sti + rec + inv + oca + fix_net + cip + disposal + intang + ltei + const_olta
        rev, pay, acc, ocl = float(debt.loc[y, "ending_revolver"]), float(wc.loc[y, "payables"]), float(wc.loc[y, "accrued_expense"]), float(wc.loc[y, "other_current_liabilities"])
        ltl, bond, oltl = float(debt.loc[y, "ending_long_term_loans"]), float(debt.loc[y, "ending_bonds"]), float(wc.loc[y, "other_long_term_liabilities"])
        total_liab = rev + pay + acc + ocl + ltl + bond + oltl
        mi = cur_mi + float(inc.loc[y, "minority_profit"]) * (1 - float(wci.loc[y, "minority_dividend_ratio"]))
        eq = cur_eq + float(inc.loc[y, "net_profit"]) - float(inc.loc[y, "cash_dividend"]) + float(shares.loc[y, "equity_issue_amount"]) + float(shares.loc[y, "debt_to_equity_reduction"])
        rows.append({"year": y, "cash_balance": cash, "short_term_investment": sti, "receivables": rec, "inventory": inv, "other_current_assets": oca, "fixed_asset_net": fix_net, "construction_in_progress": cip, "fixed_asset_disposal": disposal, "intangible_assets": intang, "long_term_equity_investment": ltei, "total_assets": total_assets, "revolver": rev, "payables": pay, "accrued_expense": acc, "other_current_liabilities": ocl, "long_term_loans": ltl, "bond_payable": bond, "other_long_term_liabilities": oltl, "total_liabilities": total_liab, "minority_interest": mi, "equity_parent_company": eq, "balance_check": total_assets - (total_liab + mi + eq)})
        cur_eq, cur_mi = eq, mi
    return pd.DataFrame(rows).set_index("year")

## 七、第二阶段汇总：生成完整预测报表

当前 notebook 只保留到财务预测，不再继续做 FCFF 折现估值。

这一部分会把前面分散计算的模块汇总起来，形成三张核心预测报表：

- 损益表预测
- 现金流量表预测
- 资产负债表预测

这样同学们即使暂时不学习估值，也可以先看清楚“历史数据 + 假设”是如何一步步变成未来财务报表的。


In [8]:
def forecast(src, payload, verbose=True):
    """完成第二阶段财务预测主流程。

    参数：
    - `src`：第一阶段源数据。
    - `payload`：完整输入字典。
    - `verbose`：是否打印日志。
    """
    title("Stage 2/2: Financial Forecasting", verbose)
    years = list(range(int(src["historical_years"][-1]) + 1, int(src["historical_years"][-1]) + 1 + int(payload["forecast_config"].get("forecast_horizon", 10))))
    rs = resolve_inputs(payload, years)
    subtitle("2.1 Forecast Horizon", verbose)
    log(f"forecast_years={years}", verbose)
    int_exp = pd.Series(0.0, index=years, dtype=float)
    int_inc = pd.Series(0.0, index=years, dtype=float)
    cap_i = pd.Series(0.0, index=years, dtype=float)
    max_iter, tol, gap = int(payload["forecast_config"].get("max_iterations", 8)), float(payload["forecast_config"].get("tolerance", 1e-6)), None
    subtitle("2.2 Debt-Cash-Interest Solver", verbose)
    for i in range(max_iter):
        shares = share_schedule(src, rs["equity"], years)
        capex = capex_schedule(src, rs, years, cap_i)
        inc = income_schedule(src, rs, capex, shares, years, int_exp, int_inc)
        wc = wc_schedule(src, rs, inc, capex, years)
        cf0 = cashflow_schedule(rs, shares, inc, wc, capex, years)
        debt = debt_schedule(src, rs, shares, cf0, capex, years)
        new_int_exp, new_int_inc, new_cap_i = debt["interest_expense"], debt["cash_interest_income"], debt["capitalized_interest"]
        gap = max((new_int_exp - int_exp).abs().max(), (new_int_inc - int_inc).abs().max(), (new_cap_i - cap_i).abs().max())
        log(f"iteration {i + 1}: gap={gap:.8f}", verbose)
        int_exp, int_inc, cap_i = new_int_exp, new_int_inc, new_cap_i
        if gap <= tol:
            log(f"converged under tolerance={tol}", verbose)
            break
    shares = share_schedule(src, rs["equity"], years)
    capex = capex_schedule(src, rs, years, cap_i)
    inc = income_schedule(src, rs, capex, shares, years, int_exp, int_inc)
    wc = wc_schedule(src, rs, inc, capex, years)
    cf0 = cashflow_schedule(rs, shares, inc, wc, capex, years)
    debt = debt_schedule(src, rs, shares, cf0, capex, years)
    bs = bs_schedule(src, rs, shares, inc, wc, capex, debt, years)
    subtitle("2.3 First-Year Trace", verbose)
    y0 = years[0]
    log(f"revenue={inc.loc[y0, 'revenue']:.4f}, ebit={inc.loc[y0, 'ebit']:.4f}, net_profit={inc.loc[y0, 'net_profit']:.4f}", verbose)
    log(f"operating_cf={cf0.loc[y0, 'operating_cf']:.4f}, investing_cf={cf0.loc[y0, 'investing_cf']:.4f}, financing_cf={cf0.loc[y0, 'financing_cf']:.4f}", verbose)
    log(f"balance_check max abs={bs['balance_check'].abs().max():.6f}", verbose)
    log("[Stage 2/2 Completed] Financial forecasting finished", verbose)
    return {
        "forecast_years": years,
        "assumptions": rs,
        "share_schedule": shares,
        "capex_schedule": capex,
        "income_statement_forecast": inc,
        "working_capital_schedule": wc,
        "cashflow_schedule": cf0,
        "debt_schedule": debt,
        "balance_sheet_schedule": bs,
        "solver_gap": gap,
    }


def run_fcff_script(user_inputs, verbose=True):
    """串联两阶段流程并返回财务预测结果。

    参数：
    - `user_inputs`：完整输入字典，通常直接使用 `USER_INPUTS`。
    - `verbose`：是否打印日志。
    """
    src = get_source(user_inputs["data_import"], verbose)
    fc = forecast(src, user_inputs, verbose)
    return {"source_data": src, "forecast_output": fc}


## 八、样例输入

下面这份 `USER_INPUTS` 是南玻A在 `2022-03-31` 的教学样例。

这里仍然保留了完整的经营、营运资本、资本开支和债务假设，目的是让三张预测报表都能顺利生成。

如果课堂上只练习报表预测，可以只修改这里的输入，而不用再关心后续估值参数。


In [9]:
USER_INPUTS = {
    "data_import": {"stock_code": "000012.SZ", "report_date": "2022-03-31", "annual_report_released": False, "lookback_years": 5, "benchmark_code": "000001.SH", "statement_scope": "合并报表", "market": "cn"},
    "forecast_config": {"forecast_horizon": 10, "max_iterations": 8, "tolerance": 1e-6},
    "forecast_assumptions": {
        "operating": {"revenue_growth": [0.088, 0.088, 0.088, 0.088, 0.088, 0.088, 0.088, 0.088, 0.088, 0.055], "operating_cost_ratio": [0.495] * 10, "ga_ratio": [0.0387, 0.0378, 0.0369, 0.0369, 0.0369, 0.0369, 0.0369, 0.0369, 0.0369, 0.0369], "rnd_ratio": [0.0314707824295231, 0.032381597279503, 0.032665427747271, 0.0321726024854323, 0.0324065425040688, 0.032414857578924, 0.032331334189475, 0.0323842447574893, 0.0323768121752961, 0.0323641303740868], "selling_ratio": [0.072] * 10, "sales_tax_ratio": [0.0198, 0.01935, 0.0189, 0.0189, 0.0189, 0.0189, 0.0189, 0.0189, 0.0189, 0.0189], "income_tax_rate": [0.225] * 10, "dividend_payout_ratio": [0.33, 0.33, 0.33, 0.33, 0.33, 0.66, 0.66, 0.66, 0.825, 0.99], "other_business_profit": [0.0] * 10, "impairment_and_fair_value": [15.0, 15.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "investment_income": [3.0] * 10, "fx_gain_loss": [0.0] * 10, "non_operating_pnl": [0.0] * 10},
        "working_capital": {"ar_days": [9.28826622681037] * 10, "inventory_days": [87.0] * 10, "other_current_asset_ratio": [0.00592455735380319] * 10, "ap_days": [26.9261100088137] * 10, "accrued_expense_ratio": [0.0] * 10, "other_current_liability_ratio": [0.120174533582733] * 10, "long_term_equity_investment_increase": [-5.53723715000001] * 10, "other_long_term_liability_increase": [-50.90027746] * 10, "minority_dividend_ratio": [0.602240700744412] * 10},
        "capex": {"fixed_asset_new_project_investment": [12.0, 62.0, 62.0, 62.0, 112.0, 142.0, 62.0, 62.0, 62.0, 62.0], "intangible_capex": [1.0] * 10, "fixed_asset_disposal": [0.0] * 10, "cip_loan_coverage_ratio": [0.0] * 10, "cip_completion_ratio": [0.85, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7], "existing_fixed_asset_remaining_life": 10, "new_fixed_asset_life": 15, "renovation_investment_ratio": [0.05] * 10, "impairment_ratio": [0.074716378750857] * 10, "intangible_amortization_life": 25, "opening_cip_interest_balance": 0.0, "renovation_transfer_ratio": [1.0] * 10},
        "debt": {"min_cash_balance": [3000.0] * 10, "cash_interest_rate": [0.025] * 10, "min_revolver_balance": [500.0, 300.0, 200.0, 200.0, 200.0, 200.0, 200.0, 200.0, 200.0, 200.0], "revolver_interest_rate": [0.045] * 10, "bond_increment": [0.0] * 10, "bond_interest_rate": [0.035] * 10, "long_term_loan_increment": [-230.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "long_term_loan_interest_rate": [0.053] * 10, "capitalized_interest_rate": [0.035] * 10},
        "equity": {"share_split_multiplier": [1.0] * 10, "equity_issue_price": [0.0] * 10, "equity_issue_shares": [0.0] * 10, "convertible_price": [0.0] * 10, "convertible_shares": [0.0] * 10},
        "asset_policy": {"short_term_investment_balance": [0.0] * 10},
    },
}


def get_user_inputs():
    """返回一份可修改的默认输入字典。

    参数：
    - 无。
    """
    return copy.deepcopy(USER_INPUTS)


## 九、运行样例并输出三张预测报表

最后这一格直接运行整个财务预测流程，并完整展示三张核心预测表：

- 损益表预测
- 现金流量表预测
- 资产负债表预测

为了便于课堂查看，这里会打开 pandas 的完整显示选项，尽量把所有列都展示出来。


In [10]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

user_inputs = get_user_inputs()
result = run_fcff_script(user_inputs, verbose=True)
forecast_output = result["forecast_output"]



Stage 1/3: Data Acquisition

[1.1 Input Context]
stock_code=000012.SZ, report_date=2022-03-31, annual_report_released=False

[1.2 Historical Window]
historical_years=[2016, 2017, 2018, 2019, 2020]

[1.3 Market Snapshot]
security_name            南玻A
close_price             7.42
total_shares_mn   3,070.6921
[Stage 1/3 Completed] Data acquisition finished

Stage 2/2: Financial Forecasting

[2.1 Forecast Horizon]
forecast_years=[2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030]

[2.2 Debt-Cash-Interest Solver]
iteration 1: gap=696.69951324
iteration 2: gap=16.51443827
iteration 3: gap=0.31262345
iteration 4: gap=0.00366703
iteration 5: gap=0.00001740
iteration 6: gap=0.00000061
converged under tolerance=1e-06

[2.3 First-Year Trace]
revenue=11610.3237, ebit=3937.0915, net_profit=2891.8224
operating_cf=3558.0825, investing_cf=-7.4628, financing_cf=-1184.3014
balance_check max abs=0.000000
[Stage 2/2 Completed] Financial forecasting finished


In [11]:

print("\n=== 损益表预测 ===")
display(forecast_output["income_statement_forecast"].T)




=== 损益表预测 ===


year,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030
revenue,"11,610.3237","12,632.0322","13,743.6511","14,953.0924","16,268.9645","17,700.6334","19,258.2891","20,953.0186","22,796.8842","24,050.7128"
operating_cost,"5,747.1103","6,252.8560","6,803.1073","7,401.7807","8,053.1374","8,761.8135","9,532.8531","10,371.7442","11,284.4577","11,905.1028"
selling_expense,835.9433,909.5063,989.5429,"1,076.6227","1,171.3654","1,274.4456","1,386.5968","1,508.6173","1,641.3757","1,731.6513"
ga_expense_ex_amort,449.3195,477.4908,507.1407,551.7691,600.3248,653.1534,710.6309,773.1664,841.2050,887.4713
ebit,"3,937.0915","4,293.1152","4,689.5742","5,113.6378","5,563.8438","6,057.3267","6,595.9917","7,179.3422","7,815.3055","8,247.9597"
pretax_profit,"3,879.0821","4,319.9318","4,786.1510","5,299.2936","5,843.7027","6,417.8555","7,023.7508","7,679.4234","8,379.2250","8,852.6668"
minority_profit,114.4663,127.4751,141.2326,156.3747,172.4394,189.3819,207.2610,226.6090,247.2591,261.2297
net_profit,"2,891.8224","3,220.4721","3,568.0344","3,950.5778","4,356.4302","4,784.4561","5,236.1459","5,724.9442","6,246.6402","6,599.5870"
cash_dividend,954.3014,"1,062.7558","1,177.4514","1,303.6907","1,437.6220","3,157.7410","3,455.8563","3,778.4632","5,153.4782","6,533.5912"


In [12]:
print("\n=== 现金流量表预测 ===")
display(forecast_output["cashflow_schedule"].T)




=== 现金流量表预测 ===


year,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030
operating_cf,"3,558.0825","4,268.0424","4,627.9027","5,014.4836","5,428.0080","5,865.8711","6,321.0709","6,813.9503","7,339.9701","7,725.0371"
investing_cf,-7.4628,-57.4628,-57.4628,-57.4628,-107.4628,-137.4628,-57.4628,-57.4628,-57.4628,-57.4628
financing_cf_pre_revolver,"-1,184.3014","-1,062.7558","-1,177.4514","-1,303.6907","-1,437.6220","-3,157.7410","-3,455.8563","-3,778.4632","-5,153.4782","-6,533.5912"
revolver_change,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
financing_cf,"-1,184.3014","-1,062.7558","-1,177.4514","-1,303.6907","-1,437.6220","-3,157.7410","-3,455.8563","-3,778.4632","-5,153.4782","-6,533.5912"
cash_net_change,"2,366.3183","3,147.8238","3,392.9886","3,653.3301","3,882.9233","2,570.6673","2,807.7518","2,978.0244","2,129.0292","1,133.9831"


In [13]:
print("\n=== 资产负债表预测 ===")
display(forecast_output["balance_sheet_schedule"].T)


=== 资产负债表预测 ===


year,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030
cash_balance,"3,711.6799","6,659.5038","9,952.4923","13,605.8224","17,488.7458","20,059.4131","22,867.1649","25,845.1894","27,974.2186","29,108.2017"
short_term_investment,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
receivables,295.4514,321.4512,349.7389,380.5159,414.0013,450.4334,490.0716,533.1978,580.1193,612.0258
inventory,"1,175.6018","1,283.3804","1,414.3205","1,560.8236","1,720.7912","1,894.6340","2,084.0913","2,290.8427","2,516.0217","2,668.9775"
other_current_assets,68.7860,74.8392,81.4250,88.5905,96.3864,104.8684,114.0968,124.1374,135.0614,142.4898
fixed_asset_net,"9,759.8780","9,067.5847","8,149.6071","7,162.8308","6,183.9634","5,222.5174","4,211.3089","3,182.7345","2,147.0325","1,107.4404"
construction_in_progress,211.6759,-76.1170,-151.2319,-158.8849,-130.1839,-96.7049,-95.0750,-78.1931,-56.4540,-33.1423
fixed_asset_disposal,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
intangible_assets,"1,095.1295","1,050.5408","1,005.9521",961.3633,916.7746,872.1859,827.5971,783.0084,738.4197,693.8310
long_term_equity_investment,-5.5372,-11.0745,-16.6117,-22.1489,-27.6862,-33.2234,-38.7607,-44.2979,-49.8351,-55.3724
